In [ ]:
Purpose:
Clean Orders Data

Operations:
1. Read Bronze Orders
2. Convert String Timestamp
3. Extract Date Components
4. Handle Null Values
5. Save Silver Layer

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    to_timestamp,
    year,
    month,
    dayofmonth,
    quarter,
    date_format
)

In [7]:
spark =( 
    SparkSession.builder
    .appName("Clean Order")
    .master("local[*]")
    .getOrCreate()
)



In [8]:
# Read Bronze Layer
orders_df = spark.read.parquet(
    "output/bronze/orders"
)

In [10]:
# Initial Inspection
print("\nOriginal Row Count:")
print(orders_df.count())

print("\nSchema:")
orders_df.printSchema()

print("\nSample Data:")
orders_df.show(5, truncate=False)



Original Row Count:
99441

Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)


Sample Data:
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+-----

In [24]:
# Remove Duplicate Records

orders_df = orders_df.dropDuplicates()

In [25]:
 #Remove Invalid Order IDs
 orders_df = orders_df.filter(col("order_id").isNotNull())

In [26]:
# Convert Purchase Timestamp

orders_df =orders_df.withColumn("purchase_timestamp",to_timestamp(col("order_purchase_timestamp")))

In [27]:
# Extract Year
orders_df = orders_df.withColumn("purchase_year",year("purchase_timestamp"))

In [28]:
# Extract Month
orders_df =orders_df.withColumn("purchase_month",month("purchase_timestamp"))

In [30]:
# Extract Day
orders_df = orders_df.withColumn("purchase_day",dayofmonth("purchase_timestamp"))

In [31]:
# Extract Quarter
orders_df=orders_df.withColumn("purchase_quarter",quarter("purchase_timestamp"))

In [32]:
# Extract Weekday Name
orders_df = orders_df.withColumn(
    "purchase_weekday",
    date_format(
        "purchase_timestamp",
        "EEEE"
    )
)

In [34]:
# Check Null Purchase Dates
null_dates = orders_df.filter(
    col("purchase_timestamp").isNull()
).count()

print(
    f"\nNull Purchase Dates: {null_dates}"
)


Null Purchase Dates: 0


In [35]:
# Display Transformed Data
# ====================================

print("\nTransformed Data:")

orders_df.select(
    "order_id",
    "purchase_timestamp",
    "purchase_year",
    "purchase_month",
    "purchase_day",
    "purchase_quarter",
    "purchase_weekday"
).show(10, truncate=False)


Transformed Data:
+--------------------------------+-------------------+-------------+--------------+------------+----------------+----------------+
|order_id                        |purchase_timestamp |purchase_year|purchase_month|purchase_day|purchase_quarter|purchase_weekday|
+--------------------------------+-------------------+-------------+--------------+------------+----------------+----------------+
|fd1b1552b93d46b554afde387322f286|2018-06-09 09:59:35|2018         |6             |9           |2               |Saturday        |
|e93c226c60a57236abb9d0962b440be9|2017-11-27 17:00:34|2017         |11            |27          |4               |Monday          |
|6646811f070a6e46f85c938eb20b1842|2017-11-26 11:01:04|2017         |11            |26          |4               |Sunday          |
|72a96400285f2bd8dd6d47e55e203b92|2018-03-24 16:36:05|2018         |3             |24          |1               |Saturday        |
|2e10c534d5d49b8fcea5e034821a96e9|2018-02-14 18:41:02|2018      